# Linha de base — a régua antes do modelo

**Capítulo 01** do livro vivo [Ciência de Dados e Aprendizado de Máquina](https://machinelearning.ghdaru.com.br/01-fundamentos.html).

Antes de treinar qualquer coisa, é preciso saber **contra o que comparar**. Este notebook constrói a linha de base — o modelo que não aprende nada — e mostra por que 81% de acurácia pode ser um resultado péssimo.

Só biblioteca padrão. Nem NumPy.

In [ ]:
# --- roda igual na sua máquina e no Colab ------------------------------
# Na sua máquina: o notebook acha o repositório subindo de pasta.
# No Colab: não há repositório, então os arquivos necessários são baixados.
import pathlib, sys, urllib.request

RAW = "https://raw.githubusercontent.com/GHDaru/machinelearning/main/"
PRECISA = ['ml-zero/etapa-00/dados.py', 'ml-zero/etapa-00/baseline.py']

raiz = pathlib.Path.cwd()
for _ in range(5):
    if (raiz / "ml-zero").is_dir():
        break
    raiz = raiz.parent
else:
    raiz = pathlib.Path.cwd()

for rel in PRECISA:
    destino = raiz / rel
    if not destino.exists():
        destino.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(RAW + rel, destino)
        print("baixado:", rel)

sys.path.insert(0, str(raiz / "ml-zero/etapa-00"))
RAIZ = raiz
print("pronto.")

## 1. O dado, dividido com cuidado

Três partes, não duas: treino, validação e teste. A divisão é **estratificada** — a proporção de classes se mantém.

In [ ]:
from dados import gerar, dividir

conjunto = gerar()
treino, validacao, teste = dividir(conjunto)

for nome, parte in [("treino", treino), ("validação", validacao), ("teste", teste)]:
    prev = sum(parte.y) / len(parte.y)
    print(f"{nome:12s} {len(parte.y):5d}  prevalência {prev:.3f}")

## 2. O modelo que não aprende nada

`MajorityBaseline` responde sempre a classe mais frequente. É o piso: qualquer modelo que não o supere não está aprendendo.

In [ ]:
from baseline import MajorityBaseline, acuracia, matriz_confusao

base = MajorityBaseline().fit(treino.X, treino.y)
pred = base.predict(validacao.X)

print(f"acurácia na validação  {acuracia(validacao.y, pred):.4f}")
print("matriz de confusão    ", matriz_confusao(validacao.y, pred))

## 3. Por que a acurácia engana

Olhe a matriz: o modelo **nunca** acerta um positivo. Ele não encontrou um único caso da classe que interessa — e mesmo assim a acurácia parece boa.

É o efeito do desbalanceamento. Com 19% de positivos, quem responde sempre "não" acerta 81% das vezes sem saber nada.

Calcule a revocação da classe 1 e veja o número que o relatório deveria mostrar:

In [ ]:
mc = matriz_confusao(validacao.y, pred)
positivos = mc["vp"] + mc["fn"]
revocacao = mc["vp"] / positivos if positivos else 0.0
print(f"positivos reais na validação: {positivos}")
print(f"encontrados pelo modelo:      {mc['vp']}")
print(f"revocação da classe 1:        {revocacao:.4f}")

## O que levar

- **Toda métrica precisa de uma régua.** 0,81 sozinho não significa nada.
- A escolha da métrica vem **antes** do treino, e sai do custo do erro (capítulo 04).
- A linha de base custa minutos e evita meses de autoengano.

Continue em [01 — Fundamentos](https://machinelearning.ghdaru.com.br/01-fundamentos.html).